# FactorEM — CryoEM Heterogeneity Analysis

## Parameters

In [1]:
PREFIX = '/home/oier/Datasets/IgG-1D/snr0.01/'
INPUT_FILE = 'snr0.01_rln4.star'
INPUT = PREFIX + '/' + INPUT_FILE

DIAMETER       = 180.0
RESOLUTION     = 4.0
PADDING_FACTOR = 2.0

APERTURE_INDEX  = 4.0
DIRECTION_INDEX = 4.0

EMBEDDING     = 'spectral'
COMPONENTS    = 6
MIN_PARTICLES = 100

DEVICE = 'gpu:0'

## Setup

In [2]:
import math

import jax
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import sklearn.decomposition
import starfile
import tqdm

from factorem import analysis, geometry, image, synchronization
from factorem.__main__ import select_device
from factorem.bsr_array_builder import BsrArrayBuilder

In [3]:
device = select_device(DEVICE)
print(f'Using device:    {device}')
print(f'JAX version:     {jax.__version__}')
print(f'Default backend: {jax.default_backend()}')

Using device:    cuda:0
JAX version:     0.10.0
Default backend: gpu


## Load Data

In [4]:
star = starfile.read(INPUT)
particles_md = star['particles']
optics       = star['optics']

pixel_size           = optics.at[0, 'rlnImagePixelSize']
amplitude_contrast   = optics.at[0, 'rlnAmplitudeContrast']
spherical_aberration = optics.at[0, 'rlnSphericalAberration']
voltage              = optics.at[0, 'rlnVoltage']
box_size             = optics.at[0, 'rlnImageSize']

print(f'Particles:            {len(particles_md)}')
print(f'Box size:             {box_size} px')
print(f'Pixel size:           {pixel_size} Å')
print(f'Voltage:              {voltage} kV')
print(f'Spherical aberration: {spherical_aberration} mm')
print(f'Amplitude contrast:   {amplitude_contrast}')

Particles:            100000
Box size:             128 px
Pixel size:           3.0 Å
Voltage:              300.0 kV
Spherical aberration: 2.7 mm
Amplitude contrast:   0.1


In [5]:
image_locations = particles_md['rlnImageName'].map(image.ImageLocation.parse)

rotations = geometry.euler_zyz_to_matrix(
    np.deg2rad(np.asarray(particles_md['rlnAngleRot'])),
    np.deg2rad(np.asarray(particles_md['rlnAngleTilt'])),
    np.deg2rad(np.asarray(particles_md['rlnAnglePsi'])),
)
shifts = (1 / pixel_size) * np.stack(
    (
        np.asarray(particles_md['rlnOriginXAngst']),
        np.asarray(particles_md['rlnOriginYAngst']),
    ),
    axis=1,
)
defocus_u = np.asarray(particles_md['rlnDefocusU'])
defocus_v = np.asarray(particles_md['rlnDefocusV'])
defocus   = 0.5 * (defocus_u + defocus_v)
image_count = len(image_locations)

## Projection Directions

In [6]:
shannon_angle      = RESOLUTION / DIAMETER
direction_spacing  = DIRECTION_INDEX * shannon_angle
direction_aperture = APERTURE_INDEX  * shannon_angle
max_freq           = pixel_size / RESOLUTION

print(f'Shannon angle:      {math.degrees(shannon_angle):.2f}°')
print(f'Direction spacing:  {math.degrees(direction_spacing):.2f}°')
print(f'Direction aperture: {math.degrees(direction_aperture):.2f}°')
print(f'Max digital freq:   {max_freq:.4f}')

direction_count    = geometry.estimate_projection_direction_count(direction_spacing)
directions         = geometry.sample_projection_directions(direction_count)
direction_matrices = geometry.euler_zyz_to_matrix(
    directions[:, 0],
    directions[:, 1],
    np.array(0),
)
groups = geometry.group_projection_directions(
    rotations[:, 2, :],
    direction_matrices[:, 2, :],
    direction_aperture,
)
for group in groups:
    group.sort()

print(f'\nDirection count: {direction_count}')

Shannon angle:      1.27°
Direction spacing:  5.09°
Direction aperture: 5.09°
Max digital freq:   0.7500

Direction count: 795


In [7]:
# Interactive 3-D scatter: each dot is a projection direction, colour = particle count
z = direction_matrices[:, 2, :]
c = [len(g) for g in groups]

fig = go.Figure(
    go.Scatter3d(
        x=z[:, 0], y=z[:, 1], z=z[:, 2],
        mode='markers',
        marker=dict(
            size=4,
            color=c,
            colorscale='Viridis',
            cmin=0,
            cmax=max(c),
            colorbar=dict(title='Particles'),
            showscale=True,
        ),
        text=[f'Dir {i}: {n} particles' for i, n in enumerate(c)],
        hovertemplate='%{text}<extra></extra>',
    )
)
fig.update_layout(
    title='Projection directions — coloured by particle count',
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

## Analysis Pipeline

In [8]:
padded_box_size = round(PADDING_FACTOR * box_size)

loader = analysis.DataLoader(
    image_locations=image_locations,
    image_prefix=PREFIX,
    rotations=rotations,
    shifts=shifts,
    defocus=defocus,
)
preprocessor = analysis.Preprocessor(
    padded_box_size=padded_box_size,
    pixel_size_a=pixel_size,
    voltage_kv=voltage,
    spherical_aberration_mm=spherical_aberration,
    amplitude_contrast=amplitude_contrast,
    max_freq=max_freq,
    grain_size=256,
)

if EMBEDDING == 'pca':
    processor = analysis.PCA(n_components=COMPONENTS, particle_size=box_size)
else:
    processor = analysis.SpectralEmbedding(
        n_components=COMPONENTS,
        kernel='median',
    )

jobs = [
    analysis.Job(key=i, indices=groups[i], direction_matrix=direction_matrices[i])
    for i in range(direction_count)
    if len(groups[i]) >= MIN_PARTICLES
]
analyzed_direction_count = len(jobs)
print(f'Directions to analyse: {analyzed_direction_count} / {direction_count}')

Directions to analyse: 795 / 795


In [1]:
from plotly.subplots import make_subplots

builder = BsrArrayBuilder((analyzed_direction_count * COMPONENTS, image_count))
runner  = analysis.PipelinedRunner(
    loader=loader,
    preprocessor=preprocessor,
    processor=processor,
    device=device,
    prefetch=4,
)

nrows = 3
ncols = 3
n_scatter = nrows*ncols
scatter_fig = make_subplots(
    rows=nrows, 
    cols=ncols, 
    subplot_titles=[' '] * n_scatter
)

progress = tqdm.tqdm(total=analyzed_direction_count, unit='dir')
for k, (job, y) in enumerate(runner.run(jobs, sequential=True)):
    i       = job.key
    indices = groups[i]
    assert np.all(indices[:-1] < indices[1:])
    for j, index in enumerate(indices):
        builder.add_block(index, np.asarray(y[j, :, None]))
    builder.next_block_row()

    if k < n_scatter:
        row, col = divmod(k, ncols)
        scatter_fig.add_trace(
            go.Scattergl(
                x=np.asarray(y[:, 0]),
                y=np.asarray(y[:, 1]),
                mode='markers',
                marker=dict(size=3),
                showlegend=False,
                customdata=np.asarray(indices),
                hovertemplate=(
                    'id %{customdata}<br>x %{x:.3f}<br>y %{y:.3f}'
                    '<extra></extra>'
                ),
            ),
            row=row + 1,
            col=col + 1,
        )
        scatter_fig.layout.annotations[k].update(text=f'Direction {i}')

    progress.update(1)
progress.close()

scatter_fig.update_layout(
    title='First few direction embeddings — component 1 vs component 2',
    height=320 * nrows,
)
scatter_fig.show()

NameError: name 'BsrArrayBuilder' is not defined

In [ ]:
print(particles_md.loc[[95360, 48525, 8449, 61080]])

                     rlnImageName  rlnDefocusU  rlnDefocusV  rlnDefocusAngle  \
95360  361@095_particles_128.mrcs    14713.100    13895.975        95.485430   
48525  526@048_particles_128.mrcs    16833.783    15899.215        93.328380   
8449   450@008_particles_128.mrcs    12558.917    11713.923        90.820610   
61080   81@061_particles_128.mrcs    14690.819    13843.803        93.262856   

       rlnVoltage  rlnSphericalAberration  rlnAmplitudeContrast  \
95360       300.0                     2.7                   0.1   
48525       300.0                     2.7                   0.1   
8449        300.0                     2.7                   0.1   
61080       300.0                     2.7                   0.1   

       rlnPhaseShift  rlnAngleRot  rlnAngleTilt  rlnAnglePsi  rlnOriginX  \
95360            0.0  -128.114895     92.320285    41.552670   -9.663631   
48525            0.0    13.286102     90.085151   -90.031688   -7.984766   
8449             0.0     9.340381  

## Synchronization

In [ ]:
embeddings   = builder.build()
similarities = embeddings @ embeddings.T
similarities /= abs(similarities).max()

In [ ]:
synchronization_transform, _ = synchronization.burer_monteiro_ortho_group_synchronization(
    similarities,
    synchronization.burer_monteiro_random_start(
        n=analyzed_direction_count,
        k=COMPONENTS,
        p=2 * COMPONENTS + 1,
    ),
)

embeddings = synchronization.correct_embeddings(
    embeddings=embeddings,
    transforms=synchronization_transform,
)

In [ ]:
similarities = embeddings @ embeddings.T
similarities /= abs(similarities).max()

## Output

In [ ]:
unified_embedding = synchronization.average_embeddings(
    embeddings=embeddings,
    max_iter=0,  # TODO: set to 16 when fixed
)

pca = sklearn.decomposition.PCA(n_components=COMPONENTS)
unified_embedding = pca.fit_transform(unified_embedding)